<div style="background:linear-gradient(135deg,#1a1a2e,#0f3460);padding:46px 36px;border-radius:14px;text-align:center">
<h1 style="color:#e94560;font-size:40px;margin:0">K-Nearest Neighbors (KNN)</h1>
<h2 style="color:#a8dadc;font-size:20px;margin-top:10px;font-weight:400">A University-Style Complete Lecture</h2>
<p style="color:#aaa;font-size:14px;margin-top:16px;line-height:1.9">
From the very basics &rarr; Distance Formulas &rarr; Classification &rarr; Regression<br>
&rarr; Weighted KNN &rarr; Decision Boundaries &rarr; Impact of K &rarr; Pros &amp; Cons &rarr; Best Practices
</p>
</div>

---

## Lecture Roadmap

| Part | Topic |
|------|-------|
| **1** | The Big Idea — what KNN is and how it thinks |
| **2** | Distance Formulas — the math of "closeness" |
| **3** | Feature Scaling — the rule you must never break |
| **4** | Classification — predicting categories |
| **5** | Regression — predicting numbers |
| **6** | Weighted KNN — closer neighbours get more say |
| **7** | Decision Boundaries — visualising what KNN learns |
| **8** | The K Problem — bias, variance, and finding the sweet spot |
| **9** | Pros and Cons — honest assessment with live demos |
| **10** | Best Practices — production-ready checklist and template |


---

## Run This First — Imports

Load every library we need before starting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.datasets import (make_classification, make_blobs,
                               make_moons, make_circles, load_iris)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from collections import Counter
import time, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize']   = (12, 6)
plt.rcParams['font.size']        = 12
plt.rcParams['axes.titlesize']   = 13
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False
sns.set_style('whitegrid')
print('All libraries loaded. Let the lecture begin!')


---

# PART 1 — The Big Idea

## 1.1 What is KNN?

Before any equations, let us start with a story.

---

### The New Neighbourhood Story

Imagine you just moved to a new city. You want to know: **is your neighbourhood safe or dangerous?**

You have no data about it yet. So what do you naturally do?

You **look at your closest neighbours** — the people living nearest to you. If most of your 5 nearest neighbours are friendly and safe-looking, you conclude: *"This seems like a safe neighbourhood."*

You did not analyse global crime statistics. You did not build a complex model. You just **looked at the K closest people around you and took a vote.**

**That is literally KNN.**

---

### Formal Definition

> **KNN (K-Nearest Neighbors)** classifies a new data point by finding the **K most similar training examples** and having them **vote** on the answer.

Three properties make KNN unique among algorithms:

| Property | What it means | Simple analogy |
|----------|--------------|----------------|
| **Non-parametric** | No assumption about the shape of the data | Does not assume a bell curve or a straight line |
| **Lazy Learner** | Zero computation during training — just memorises data | A student who only reads the textbook and never summarises |
| **Instance-based** | Every training point is stored and used at prediction time | Keeps every page of notes for every exam |

---

### The Algorithm in Plain English

```
Given a new unknown point X:

  Step 1  Calculate the distance from X to every training point
  Step 2  Sort all training points from closest to farthest
  Step 3  Take the top K (the K nearest neighbours)
  Step 4a Classification  ->  let them VOTE  ->  majority class wins
  Step 4b Regression      ->  AVERAGE their values  ->  that is the prediction
```

The graph below shows exactly this process on a small toy dataset.
The three panels walk through each step visually.


In [ ]:
# ── Step-by-step KNN voting visual ───────────────────────────────────────────
train_pts = np.array([
    [1.0, 2.0], [1.5, 1.8], [2.0, 3.0], [2.5, 2.0],
    [5.0, 5.0], [6.0, 5.5], [5.5, 6.0], [6.5, 4.5],
    [3.5, 4.0], [4.0, 3.5]
])
train_labels = ['A','A','A','A','B','B','B','B','C','C']
cmap_cls = {'A':'#e74c3c','B':'#2980b9','C':'#27ae60'}
new_pt   = np.array([3.8, 3.8])

distances  = np.linalg.norm(train_pts - new_pt, axis=1)
sorted_idx = np.argsort(distances)
K          = 3
k_idx      = sorted_idx[:K]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
fig.suptitle('How KNN Works - One Step at a Time', fontsize=17, fontweight='bold', y=1.02)

titles = [
    'Step 1: Training Data + New Mystery Point',
    'Step 2: Measure Distance to ALL Points',
    'Step 3: Pick K=3 Nearest -> Majority Vote'
]

for step, ax in enumerate(axes):
    for i, (pt, lbl) in enumerate(zip(train_pts, train_labels)):
        is_nbr = (step == 2 and i in k_idx)
        sz = 260 if is_nbr else 120
        ec = 'gold' if is_nbr else '#333'
        lw = 3     if is_nbr else 0.8
        if step == 1:
            ax.plot([new_pt[0], pt[0]], [new_pt[1], pt[1]],
                    color='#bdc3c7', lw=1, alpha=0.5, zorder=0)
            if i < 4:
                mid = (new_pt + pt) / 2
                ax.text(mid[0], mid[1], f'{distances[i]:.1f}',
                        fontsize=7, color='#555', ha='center',
                        bbox=dict(fc='white', alpha=0.6, pad=1, ec='none'))
        if step == 2 and i in k_idx:
            ax.plot([new_pt[0], pt[0]], [new_pt[1], pt[1]],
                    color='gold', lw=2.5, zorder=1, ls='--')
        ax.scatter(*pt, color=cmap_cls[lbl], edgecolors=ec,
                   s=sz, linewidths=lw, zorder=3)
        ax.text(pt[0]+0.15, pt[1]+0.15, lbl,
                fontsize=10, color=cmap_cls[lbl], fontweight='bold')

    ax.scatter(*new_pt, color='black', marker='*', s=500, zorder=5)
    ax.text(new_pt[0]+0.1, new_pt[1]+0.25, '? (NEW)',
            fontsize=11, fontweight='bold', color='black')

    if step == 2:
        votes  = Counter([train_labels[i] for i in k_idx])
        winner = votes.most_common(1)[0][0]
        vote_str = '  '.join(f'Class {c}: {v}' for c,v in votes.items())
        ax.set_title(f'{titles[step]}\n{vote_str}  ->  Predicted: Class {winner}',
                     fontsize=10, fontweight='bold', color='#27ae60')
    else:
        ax.set_title(titles[step], fontsize=11, fontweight='bold')

    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
    ax.set_xlim(0, 8);          ax.set_ylim(0.5, 7.5)
    ax.grid(True, alpha=0.25)

patches = [mpatches.Patch(color=v, label=f'Class {k}') for k,v in cmap_cls.items()]
axes[0].legend(handles=patches, fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('01_knn_steps.png', dpi=120, bbox_inches='tight')
plt.show()
print('Panel 1: Raw data. Star = mystery point.')
print('Panel 2: Lines show distances; only a few values labelled for clarity.')
print('Panel 3: Gold = 3 nearest neighbours. They vote -> majority class wins.')


---

# PART 2 — Distance Formulas

## 2.1 The Heart of KNN

Here is a critical insight that most beginners miss:

> **KNN is only as good as the distance formula you use.**

If you measure closeness in the wrong way, you will pick the wrong neighbours, and your predictions will be wrong — no matter how good your data is.

Think of it this way. Suppose you want to find your nearest friends:
- By **straight-line distance** (Euclidean) -> as the crow flies
- By **road distance** (Manhattan) -> actual travel on a grid
- By **how similarly you think** (Cosine) -> angle between your opinion vectors

Each gives a completely different answer of who is "nearest." Same idea in KNN.

---

### Formula 1 — Euclidean Distance: "As the Crow Flies"

The straight-line distance between two points. It is just the Pythagorean theorem extended to N dimensions.

$$d = \sqrt{\sum_{i=1}^{n}(A_i - B_i)^2}$$

**Use when:** Features are continuous and on similar numeric scales.
**Weakness:** Large-valued features dominate. If salary is 0-100000 and age is 0-60, salary controls everything.

---

### Formula 2 — Manhattan Distance: "The City Taxi"

Imagine a grid city. A taxi cannot go diagonally — only up/down and left/right. The total road distance is the Manhattan distance.

$$d = \sum_{i=1}^{n} |A_i - B_i|$$

**Use when:** Data lives on a grid, or when outliers are present.
**Why more robust?** Euclidean squares the differences, making large outliers explode. Manhattan just adds them — much calmer.

---

### Formula 3 — Minkowski Distance: "The Master Formula"

This generalises both Euclidean and Manhattan under one formula:

$$d = \left(\sum_{i=1}^{n} |A_i - B_i|^p \right)^{1/p}$$

- p = 1 gives Manhattan
- p = 2 gives Euclidean
- p -> infinity gives Chebyshev (only the maximum difference matters)

You can tune `p` as a hyperparameter!

---

### Formula 4 — Cosine Distance: "Thinking Alike"

Instead of measuring how far two points are, cosine similarity measures the **angle between them**. Two documents about AI point in the same direction even if one is 10 pages and the other is 100 pages.

$$d_{cosine} = 1 - \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

**Use when:** Text data, recommendation systems, high-dimensional sparse data.

---

The graph below gives you geometric intuition for these formulas.


In [ ]:
# ── Distance formulas: geometric intuition ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Distance Formulas - Geometric Intuition', fontsize=16, fontweight='bold')

A = np.array([1.0, 1.0])
B = np.array([5.0, 4.0])

# Panel 1: Euclidean vs Manhattan on a grid
ax = axes[0]
ax.set_facecolor('#f8f9fa')
for v in range(0, 8): ax.axvline(v, color='#dee2e6', lw=0.7)
for v in range(0, 7): ax.axhline(v, color='#dee2e6', lw=0.7)

eu = np.linalg.norm(B - A)
mn = np.sum(np.abs(B - A))
ax.plot([A[0],B[0]], [A[1],B[1]], '#2ecc71', lw=3,
        label=f'Euclidean = {eu:.2f}  (straight line)')
ax.plot([A[0],B[0],B[0]], [A[1],A[1],B[1]], '#e67e22', lw=3, ls='--',
        label=f'Manhattan = {mn:.0f}  (grid path)')
ax.scatter(*A, color='#3498db', s=160, zorder=5, edgecolors='k', lw=1.5)
ax.scatter(*B, color='#e74c3c', s=160, zorder=5, edgecolors='k', lw=1.5)
ax.text(A[0]-0.5, A[1]-0.35, 'A (1,1)', fontsize=11, color='#3498db', fontweight='bold')
ax.text(B[0]+0.1, B[1]+0.2,  'B (5,4)', fontsize=11, color='#e74c3c', fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_title('Euclidean vs Manhattan\n"Straight line" vs "Taxi route on a grid"', fontsize=11)
ax.set_xlim(0,7); ax.set_ylim(0,6)
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# Panel 2: Minkowski unit balls
ax = axes[1]
theta  = np.linspace(0, 2*np.pi, 400)
center = np.array([3.0, 3.0])
R      = 2.0
for p, col, ls, lbl in [(1,'#e74c3c','-','p=1  Manhattan'),
                         (2,'#3498db','-','p=2  Euclidean'),
                         (4,'#9b59b6','--','p=4  Minkowski'),
                         (30,'#27ae60',':','p->inf  Chebyshev')]:
    cp = np.cos(theta); sp = np.sin(theta)
    xp = np.sign(cp)*np.abs(cp)**(2/p)
    yp = np.sign(sp)*np.abs(sp)**(2/p)
    ax.plot(center[0]+R*xp, center[1]+R*yp, color=col, lw=2.5, ls=ls, label=lbl)
ax.scatter(*center, color='black', s=130, zorder=5)
ax.text(center[0]+0.1, center[1]+0.15, 'Origin', fontsize=10)
ax.annotate('All shapes = points\nat equal distance\nfrom centre',
            xy=(4.5,1.2), fontsize=9, color='#333',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
ax.set_title('Minkowski Unit Balls\n"Shape changes as p increases"', fontsize=11)
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(0,6); ax.set_ylim(0,6); ax.set_aspect('equal')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.3)

# Panel 3: numerical comparison bar chart
ax = axes[2]
pairs = [(np.array([0,0]), np.array([3,4])),
         (np.array([0,0]), np.array([1,1])),
         (np.array([2,2]), np.array([5,6]))]
xlabels = ['A(0,0)->B(3,4)', 'A(0,0)->B(1,1)', 'A(2,2)->B(5,6)']
bw = 0.25; xp = np.arange(len(pairs))
eu_v = [np.linalg.norm(b-a)         for a,b in pairs]
mn_v = [np.sum(np.abs(b-a))         for a,b in pairs]
ch_v = [np.max(np.abs(b-a))         for a,b in pairs]
ax.bar(xp-bw, eu_v, bw, label='Euclidean', color='#3498db', alpha=0.85)
ax.bar(xp,    mn_v, bw, label='Manhattan', color='#e67e22', alpha=0.85)
ax.bar(xp+bw, ch_v, bw, label='Chebyshev', color='#27ae60', alpha=0.85)
for xs, vs in [(xp-bw,eu_v),(xp,mn_v),(xp+bw,ch_v)]:
    for x,v in zip(xs,vs):
        ax.text(x, v+0.05, f'{v:.1f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
ax.set_xticks(xp); ax.set_xticklabels(xlabels, fontsize=9)
ax.set_title('Same Points, Different Formulas\nDifferent values = different neighbours picked!', fontsize=11)
ax.set_ylabel('Distance Value')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('02_distance_formulas.png', dpi=120, bbox_inches='tight')
plt.show()
print('Left panel: Euclidean goes diagonally; Manhattan follows the grid roads.')
print('Middle: As p increases, the equal-distance "shape" goes from diamond to square.')
print('Right:  Exact same points but each formula gives a DIFFERENT number.')
print('        This means different formulas may pick different nearest neighbours!')


---

# PART 3 — Feature Scaling

## 3.1 The Rule You Must Never Break

This is the most common beginner mistake with KNN. It will silently destroy your model.

**The problem:** Distance formulas treat all features equally by default.
If Feature 1 is `age` (range 20-60) and Feature 2 is `salary` (range 30,000-120,000),
then salary differences are thousands of times larger than age differences.

KNN will effectively **ignore age completely** because salary swamps every distance calculation.

**The fix:** Scale your features so they live on comparable ranges.

- `StandardScaler`: subtracts the mean, divides by standard deviation. Centers data around 0.
- `MinMaxScaler`: squashes everything into [0, 1].

**Critical rule:** Fit the scaler on **training data only**, then apply it to test data.
If you fit on test data too, you are leaking future information into your model.

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit AND transform
X_test_scaled  = scaler.transform(X_test)         # transform ONLY
```


In [ ]:
# ── Feature scaling: why it matters enormously ───────────────────────────────
np.random.seed(42)
X_raw, y_sc = make_moons(n_samples=400, noise=0.25, random_state=42)

X_unscaled         = X_raw.copy()
X_unscaled[:, 0]  *= 1000          # Feature 1 now in thousands
X_scaled           = StandardScaler().fit_transform(X_unscaled)

k_vals   = range(1, 26)
acc_raw  = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                            X_unscaled, y_sc, cv=5).mean() for k in k_vals]
acc_sc   = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                            X_scaled,   y_sc, cv=5).mean() for k in k_vals]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Feature Scaling - Why It Matters Enormously for KNN',
             fontsize=15, fontweight='bold')

axes[0].plot(k_vals, acc_raw, 'r-o', ms=5, lw=2,
             label='NOT scaled (Feature 1 is 1000x bigger)')
axes[0].plot(k_vals, acc_sc,  'g-s', ms=5, lw=2,
             label='Scaled with StandardScaler')
axes[0].fill_between(k_vals, acc_raw, acc_sc, alpha=0.12, color='blue',
                     label='Accuracy gap')
axes[0].set_xlabel('K Value'); axes[0].set_ylabel('Cross-Validation Accuracy')
axes[0].set_title('Accuracy: Scaled vs Unscaled')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_unscaled[y_sc==0,0], X_unscaled[y_sc==0,1],
                c='#3498db', alpha=0.5, s=20, label='Class 0')
axes[1].scatter(X_unscaled[y_sc==1,0], X_unscaled[y_sc==1,1],
                c='#e74c3c', alpha=0.5, s=20, label='Class 1')
axes[1].set_title('Unscaled Data — Feature 1 dominates completely\n'
                  '(data is a thin vertical strip; distances are meaningless)', fontsize=11)
axes[1].set_xlabel('Feature 1 (x1000 scale)'); axes[1].set_ylabel('Feature 2 (normal)')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].text(0.5, 0.04,
             'KNN uses Feature 1 only and ignores Feature 2 entirely',
             transform=axes[1].transAxes, ha='center', fontsize=10,
             color='darkred',
             bbox=dict(fc='#fff3cd', ec='orange', alpha=0.9, pad=5))

plt.tight_layout()
plt.savefig('03_feature_scaling.png', dpi=120, bbox_inches='tight')
plt.show()
print('Rule: ALWAYS scale features before KNN. No exceptions.')


---

# PART 4 — KNN for Classification

## 4.1 Predicting Categories

Classification means predicting a **label** — spam or not spam, cat or dog, which disease.

KNN classification works like this:

> Find K nearest neighbours -> count how many belong to each class -> predict the class with the most votes.

Example with K=5 and neighbours = [Cat, Dog, Cat, Cat, Dog]:
- Cat: 3 votes, Dog: 2 votes -> **Prediction: Cat**

That simple idea works remarkably well in practice.

### What is a Training/Test Split?

You never evaluate a model on the data it was trained on — that is cheating.
Instead, you split your data:
- **Training set (80%)**: the model learns from this
- **Test set (20%)**: completely unseen data used to evaluate real performance

### What is a Confusion Matrix?

After predicting on the test set, a confusion matrix shows exactly where the model got confused.
Each row is the **true class**, each column is the **predicted class**.
- Numbers on the diagonal = correct predictions
- Numbers off the diagonal = mistakes (which class was confused with which)


In [ ]:
# ── KNN Classification on Iris ───────────────────────────────────────────────
iris    = load_iris()
X_iris  = iris.data
y_iris  = iris.target
cnames  = iris.target_names

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris,
                                            test_size=0.2, random_state=42)
sc      = StandardScaler()
Xtr_sc  = sc.fit_transform(X_tr)
Xte_sc  = sc.transform(X_te)

knn5    = KNeighborsClassifier(n_neighbors=5)
knn5.fit(Xtr_sc, y_tr)
y_pred5 = knn5.predict(Xte_sc)
acc5    = accuracy_score(y_te, y_pred5)

print('=' * 52)
print('  KNN Classification - Iris Dataset (K=5)')
print('=' * 52)
print(f'  Training samples : {len(X_tr)}')
print(f'  Testing  samples : {len(X_te)}')
print(f'  Accuracy         : {acc5*100:.1f}%')
print()
print(classification_report(y_te, y_pred5, target_names=cnames, digits=3))


In [ ]:
# ── Confusion matrix + accuracy vs K ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('KNN Classification Analysis - Iris Dataset',
             fontsize=16, fontweight='bold')

cm = confusion_matrix(y_te, y_pred5)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cnames, yticklabels=cnames,
            ax=axes[0], linewidths=1, linecolor='white',
            annot_kws={'size':15})
axes[0].set_title('Confusion Matrix (K=5)\n'
                  'Rows = True Class  |  Columns = Predicted Class', fontsize=12)
axes[0].set_xlabel('Predicted Label'); axes[0].set_ylabel('True Label')
axes[0].text(3.15, 1.5, 'Diagonal =\nCorrect\n\nOff-diagonal\n= Mistakes',
             fontsize=10, bbox=dict(boxstyle='round', fc='lightyellow',
                                    alpha=0.9, ec='orange'))

k_range = range(1, 31)
tr_acc = []; te_acc = []
for k in k_range:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(Xtr_sc, y_tr)
    tr_acc.append(m.score(Xtr_sc, y_tr))
    te_acc.append(m.score(Xte_sc,  y_te))

best_k = list(k_range)[np.argmax(te_acc)]
axes[1].plot(k_range, tr_acc, 'b-o', ms=5, lw=2, label='Train Accuracy')
axes[1].plot(k_range, te_acc, 'r-s', ms=5, lw=2, label='Test Accuracy')
axes[1].axvline(x=best_k, color='green', ls='--', lw=2.5,
                label=f'Best K={best_k}  (Test: {max(te_acc):.1%})')
axes[1].fill_between(k_range, tr_acc, te_acc, alpha=0.1, color='gray',
                     label='Train-Test Gap')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs K\nSmall K=overfit  |  Large K=underfit', fontsize=12)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.78, 1.02)

plt.tight_layout()
plt.savefig('04_classification.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best K = {best_k}.')
print('Confusion matrix: perfect model has numbers only on the diagonal.')


---

# PART 5 — KNN for Regression

## 5.1 When the Answer is a Number

Not everything is a category. Sometimes we predict a continuous value:
- The **price** of a house
- A patient's **blood pressure**
- Tomorrow's **temperature**

For regression, KNN uses the exact same process — find K nearest neighbours — but instead of voting, it **averages** their target values:

$$\hat{y} = \frac{1}{K} \sum_{i \in \text{K neighbours}} y_i$$

**Concrete example:** Predicting house price for a 1500 sq ft house.
With K=3, suppose the 3 nearest training houses are priced at 210k, 225k, 218k.
Prediction = (210k + 225k + 218k) / 3 = **217,667**.

### What the Graph Shows

We test KNN regression on a noisy sine wave (the dashed green line is the "true" underlying function):

- **K=1** — The prediction curve passes through every single training point including noisy ones. Train error = 0 but this will fail completely on new data. This is overfitting.
- **K=7** — Smooth, sensible fit. Follows the real curve without chasing individual noise blips.
- **K=25** — The model averages over so many neighbours that it flattens out and loses the shape of the sine wave. This is underfitting.


In [ ]:
# ── KNN Regression ───────────────────────────────────────────────────────────
np.random.seed(42)
X_reg  = np.sort(np.random.uniform(0, 10, 90)).reshape(-1, 1)
y_reg  = np.sin(X_reg).ravel() + 0.3 * np.random.randn(90)
X_line = np.linspace(0, 10, 600).reshape(-1, 1)
y_true = np.sin(X_line).ravel()

configs = [
    (1,  'K=1  - Extreme Overfitting',
         'Memorises every point and its noise.\nTrain error = 0, but will fail on new data.'),
    (7,  'K=7  - Good Balance',
         'Follows the true curve well\nwithout chasing noise.'),
    (25, 'K=25 - Underfitting',
         'Averages too broadly.\nLoses the shape of the curve.'),
]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
fig.suptitle('KNN Regression - How K Controls the Prediction Curve',
             fontsize=16, fontweight='bold')

for ax, (k, title, note) in zip(axes, configs):
    r = KNeighborsRegressor(n_neighbors=k)
    r.fit(X_reg, y_reg)
    y_hat = r.predict(X_line)
    mse   = np.mean((r.predict(X_reg) - y_reg)**2)

    ax.scatter(X_reg, y_reg, color='#7f8c8d', alpha=0.5, s=35, label='Training data')
    ax.plot(X_line, y_true, '#2ecc71', lw=2, ls='--', alpha=0.8, label='True function')
    ax.plot(X_line, y_hat,  '#e74c3c', lw=2.5, label=f'KNN K={k}')
    ax.set_title(f'{title}\nTrain MSE = {mse:.4f}', fontsize=11, fontweight='bold')
    ax.text(0.5, 0.04, note, transform=ax.transAxes, ha='center', fontsize=9.5,
            bbox=dict(fc='#fff3cd', ec='orange', alpha=0.9, pad=5))
    ax.set_xlabel('X'); ax.set_ylabel('y')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('05_regression.png', dpi=120, bbox_inches='tight')
plt.show()
print('K=1:  Train MSE=0 (perfect on training) but the curve is jagged nonsense.')
print('K=7:  Small MSE AND follows the true sine shape. This generalises well.')
print('K=25: Higher MSE because it is too smooth to capture the curve.')


---

# PART 6 — Weighted KNN

## 6.1 Not All Neighbours Are Equal

In standard KNN, all K neighbours get **one equal vote** regardless of how close they are.

But consider this scenario: you ask 5 people whether you will enjoy a new film.
- **Person A** is your best friend who knows your taste perfectly — lives next door
- **Person B** is a very distant acquaintance you met once — lives 50 streets away

Should they have equal influence? Intuitively, **no.**

**Weighted KNN** gives each neighbour a vote proportional to how close they are:

$$\text{Weight of neighbour } i = \frac{1}{d_i}$$

Closer = smaller distance = **larger weight** = more influence on the prediction.
Farther = larger distance = **smaller weight** = less influence.

For regression, the weighted average becomes:

$$\hat{y} = \frac{\displaystyle\sum_{i=1}^{K} \frac{1}{d_i} \cdot y_i}{\displaystyle\sum_{i=1}^{K} \frac{1}{d_i}}$$

In sklearn this is simply: `KNeighborsClassifier(weights='distance')`

### When Does Weighting Help Most?

- When some of your K neighbours are **much closer** than others
- Near **class boundaries** where the closest neighbours are the most informative
- With **noisy data** where distant neighbours may be from the wrong region

The right panel of the graph below shows a concrete example where uniform KNN gets the **wrong answer** but weighted KNN gets it **right** — because 2 very close Class 1 points outweigh 3 farther Class 0 points in distance-weighted voting.


In [ ]:
# ── Weighted vs Uniform KNN ───────────────────────────────────────────────────
np.random.seed(0)
X_w, y_w = make_moons(n_samples=500, noise=0.35, random_state=0)
X_w_sc   = StandardScaler().fit_transform(X_w)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(X_w_sc, y_w,
                                               test_size=0.3, random_state=0)
k_vals_w      = range(1, 31)
acc_uni, acc_dst = [], []
for k in k_vals_w:
    mu = KNeighborsClassifier(n_neighbors=k, weights='uniform')
    md = KNeighborsClassifier(n_neighbors=k, weights='distance')
    mu.fit(Xw_tr, yw_tr); md.fit(Xw_tr, yw_tr)
    acc_uni.append(mu.score(Xw_te, yw_te))
    acc_dst.append(md.score(Xw_te, yw_te))

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle('Weighted KNN vs Standard (Uniform) KNN', fontsize=16, fontweight='bold')

axes[0].plot(k_vals_w, acc_uni, 'b-o', ms=5, lw=2.5,
             label='Uniform: every neighbour = 1 vote')
axes[0].plot(k_vals_w, acc_dst, 'r-s', ms=5, lw=2.5,
             label='Distance: closer neighbours count more')
axes[0].fill_between(k_vals_w, acc_uni, acc_dst,
                     where=[d > u for d, u in zip(acc_dst, acc_uni)],
                     alpha=0.15, color='green', label='Weighted KNN wins here')
axes[0].set_xlabel('K Value'); axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Distance-weighted KNN usually wins,\nespecially at larger K values', fontsize=11)
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

# Right panel: concrete example where weighting changes the answer
ax2 = axes[1]
test_pt  = np.array([0.0,  0.0])
nbrs_pos = np.array([[ 0.20,  0.10],  # very close, Class 1
                     [ 0.30, -0.15],  # very close, Class 1
                     [-1.50,  1.20],  # far,        Class 0
                     [-1.80, -1.30],  # far,        Class 0
                     [ 1.90,  1.50]]) # far,        Class 0
nbrs_cls = [1, 1, 0, 0, 0]
nbrs_col = ['#e74c3c' if c else '#3498db' for c in nbrs_cls]
dists    = np.linalg.norm(nbrs_pos - test_pt, axis=1)
weights  = 1.0 / dists
weights /= weights.sum()

for pt, col, d, wt in zip(nbrs_pos, nbrs_col, dists, weights):
    ax2.scatter(*pt, color=col, s=200, zorder=4, edgecolors='k', lw=1.5)
    ax2.plot([test_pt[0], pt[0]], [test_pt[1], pt[1]],
             color='#bdc3c7', lw=1.5, zorder=1)
    ax2.text(pt[0]+0.06, pt[1]+0.08, f'd={d:.2f}\nw={wt:.1%}',
             fontsize=8.5, color='#333')

ax2.scatter(*test_pt, color='black', marker='*', s=500, zorder=5)
ax2.text(0.06, -0.22, 'Test Point', fontsize=11, fontweight='bold')

w1 = sum(w for c, w in zip(nbrs_cls, weights) if c == 1)
w0 = sum(w for c, w in zip(nbrs_cls, weights) if c == 0)
ax2.set_title(
    f'Why Distance Weighting Changes the Answer\n'
    f'Uniform vote -> Class 0 (3 votes)  |  '
    f'Weighted vote -> Class {"1" if w1>w0 else "0"} (weight {max(w0,w1):.0%})',
    fontsize=10)
ax2.text(0.5, 0.02,
         f'Class 0 (far, 3 points): combined weight = {w0:.0%}\n'
         f'Class 1 (near, 2 points): combined weight = {w1:.0%}',
         transform=ax2.transAxes, ha='center', fontsize=9.5,
         bbox=dict(fc='lightyellow', ec='orange', alpha=0.9, pad=5))
ax2.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Class 1 (near)'),
    mpatches.Patch(color='#3498db', label='Class 0 (far)')], fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('06_weighted_knn.png', dpi=120, bbox_inches='tight')
plt.show()
print('Right panel: Uniform KNN picks Class 0 (3 votes vs 2).')
print('Weighted KNN picks Class 1 because those 2 points are MUCH closer.')
print('The 3 Class-0 points are far away and their combined weight is lower.')


---

# PART 7 — Decision Boundaries

## 7.1 What is a Decision Boundary?

A **decision boundary** is the imaginary line (or curve) separating one class from another in feature space.
Any new point on one side gets classified as Class A; the other side gets Class B.

Think of it as a country border on a map. Which side you are on determines your passport (class).

### Why KNN Boundaries Are Special

Most classic algorithms impose a fixed shape:
- **Logistic Regression** always draws a straight line
- **Linear SVM** always draws a straight hyperplane

KNN makes **no such assumption**. Its boundaries are as complex or as simple as the data demands.
This makes KNN extremely powerful for messy, real-world data.

### How to Read the Graphs Below

The coloured background is KNN's "decision region" — the colour tells you which class KNN would predict for any new point landing in that region.
The white contour lines are the actual boundaries between regions.

We test on three dataset shapes to show KNN's flexibility:
- **Blobs**: Well-separated clusters -> clean curved boundaries
- **Moons**: Crescent shapes -> KNN wraps around each moon perfectly
- **Circles**: Concentric rings -> KNN identifies the inner circle as a separate class

Logistic Regression cannot separate the Circles dataset at all — it can only draw a straight line.


In [ ]:
# ── Decision boundary helper ──────────────────────────────────────────────────
def plot_boundary(ax, X, y, clf, title, subtitle='', res=0.025):
    n_cls   = len(np.unique(y))
    bg_cols = ['#FFCCCC','#CCCCFF','#CCFFCC'][:n_cls]
    dt_cols = ['#cc0000','#0000cc','#008800'][:n_cls]
    x0min, x0max = X[:,0].min()-0.6, X[:,0].max()+0.6
    x1min, x1max = X[:,1].min()-0.6, X[:,1].max()+0.6
    xx, yy = np.meshgrid(np.arange(x0min, x0max, res),
                         np.arange(x1min, x1max, res))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.38, cmap=ListedColormap(bg_cols))
    ax.contour(xx, yy, Z, colors='white', linewidths=1.5, alpha=0.7)
    for ci, col in enumerate(dt_cols):
        ax.scatter(X[y==ci,0], X[y==ci,1], c=col,
                   edgecolors='white', lw=0.4, s=55, zorder=3)
    acc = clf.score(X, y)
    ax.set_title(f'{title}\n{subtitle}  |  Accuracy: {acc:.1%}', fontsize=11)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# ── Three dataset shapes ──────────────────────────────────────────────────────
np.random.seed(42)
datasets = [
    (make_blobs(n_samples=250, centers=2, random_state=42, cluster_std=1.2)[:2],
     'Linearly Separable Blobs', 'Simple, well-separated clusters'),
    (make_moons(n_samples=250, noise=0.2, random_state=42),
     'Moon-Shaped Data', 'Non-linear - no straight line works here'),
    (make_circles(n_samples=250, noise=0.12, factor=0.45, random_state=42),
     'Concentric Circles', 'Linear models completely fail on this'),
]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
fig.suptitle('KNN Decision Boundaries on Different Data Shapes  (K=7)',
             fontsize=16, fontweight='bold')
for ax, ((Xd, yd), title, sub) in zip(axes, datasets):
    Xsc = StandardScaler().fit_transform(Xd)
    clf = KNeighborsClassifier(n_neighbors=7)
    clf.fit(Xsc, yd)
    plot_boundary(ax, Xsc, yd, clf, title, sub)
plt.tight_layout()
plt.savefig('07_boundaries_shapes.png', dpi=120, bbox_inches='tight')
plt.show()
print('Blobs:   Clean curved boundary between two clusters.')
print('Moons:   Boundary wraps around each crescent.')
print('Circles: Boundary forms a ring. No straight-line classifier could do this.')


---

# PART 8 — The K Problem

## 8.1 How One Number Changes Everything

K is the single most important hyperparameter in KNN. It controls the entire character of the model.

**When K = 1 (too small):**
"I will trust only my single closest training example."
This sounds precise, but it means every outlier and mislabelled point in training will corrupt nearby predictions. The decision boundary becomes an erratic mess of tiny islands.

**When K = n (all training data):**
"I will look at every training example."
Every prediction becomes the majority class of the entire dataset — completely ignoring where the new point is. Useless.

**The sweet spot:**
Large enough to average out noise, small enough to stay sensitive to local patterns.

---

### The Bias-Variance Tradeoff

This is one of the most important concepts in all of machine learning:

| K value | Bias | Variance | Effect |
|---------|------|----------|--------|
| Very small (K=1) | Low | High | **Overfitting**: wiggly, unstable boundary |
| Medium | Balanced | Balanced | **Good generalisation** |
| Very large | High | Low | **Underfitting**: boundary too smooth, misses patterns |

- **Bias** = how much the model misses the real pattern (systematic error)
- **Variance** = how much predictions change when trained on different data (sensitivity to noise)

A useful starting point: begin with $K = \sqrt{n_{\text{training}}}$, then refine using cross-validation.

The graph below shows six K values on the same noisy dataset so you can see the boundary evolution.


In [ ]:
# ── K value impact on decision boundaries ─────────────────────────────────────
np.random.seed(0)
X_km, y_km = make_moons(n_samples=350, noise=0.28, random_state=0)
X_km_sc    = StandardScaler().fit_transform(X_km)

k_cfgs = [
    (1,   'K=1   - Severe Overfitting',
           'Boundary hugs every single point.\nEvery outlier creates its own island.'),
    (3,   'K=3   - Still Overfit',
           'Slightly smoother but still very noisy.\nToo sensitive to individual points.'),
    (7,   'K=7   - Sweet Spot',
           'Captures the moon shape well\nwithout following every noise blip.'),
    (15,  'K=15  - Still Good',
           'A bit smoother than K=7\nbut still accurate on moons.'),
    (45,  'K=45  - Underfitting',
           'Getting too smooth.\nStarting to lose the moon curve.'),
    (130, 'K=130 - Severe Underfitting',
           'Nearly a straight line.\nCompletely lost the pattern.'),
]
fig, axes = plt.subplots(2, 3, figsize=(19, 12))
fig.suptitle('Impact of K on Decision Boundary - Same Data, Different K Values',
             fontsize=16, fontweight='bold')
for ax, (k, title, note) in zip(axes.ravel(), k_cfgs):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_km_sc, y_km)
    plot_boundary(ax, X_km_sc, y_km, clf, title, note)
plt.tight_layout()
plt.savefig('08_k_impact.png', dpi=120, bbox_inches='tight')
plt.show()
print('K=1:   100% train accuracy but chaotic boundary - memorised every noise point.')
print('K=7/15: Clean moon-shaped boundary - best generalisation.')
print('K=130:  Boundary is almost straight - lost all the structure of the data.')


In [ ]:
# ── Bias-Variance tradeoff curve ──────────────────────────────────────────────
np.random.seed(42)
X_bv, y_bv = make_moons(n_samples=600, noise=0.3, random_state=42)
X_bv_sc    = StandardScaler().fit_transform(X_bv)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bv_sc, y_bv,
                                               test_size=0.3, random_state=42)
k_full  = range(1, 61)
tr_err  = []
te_err  = []
for k in k_full:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(Xb_tr, yb_tr)
    tr_err.append(1 - m.score(Xb_tr, yb_tr))
    te_err.append(1 - m.score(Xb_te, yb_te))
best_k_bv = list(k_full)[np.argmin(te_err)]

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(list(k_full), tr_err, 'b-o', ms=4, lw=2.5, alpha=0.85,
        label='Training Error  (rises as K increases = more bias)')
ax.plot(list(k_full), te_err, 'r-s', ms=4, lw=2.5, alpha=0.85,
        label='Test Error  (rises at large K = underfitting)')
ax.axvline(x=best_k_bv, color='#27ae60', ls='--', lw=2.5,
           label=f'Best K = {best_k_bv}  (minimum test error)')
ax.axvspan(1, 6,    alpha=0.07, color='red')
ax.axvspan(35, 60,  alpha=0.07, color='blue')
ax.axvspan(6, 35,   alpha=0.04, color='green')
ax.text(3,   max(te_err)*0.88, 'OVERFIT\nHigh Variance\nK too small',
        ha='center', fontsize=10, color='#c0392b',
        bbox=dict(boxstyle='round', fc='#fadbd8', alpha=0.9))
ax.text(47,  max(te_err)*0.88, 'UNDERFIT\nHigh Bias\nK too large',
        ha='center', fontsize=10, color='#2980b9',
        bbox=dict(boxstyle='round', fc='#d6eaf8', alpha=0.9))
ax.text(best_k_bv, min(te_err)-0.012, f'Sweet Spot\nK={best_k_bv}',
        ha='center', fontsize=10, color='#27ae60', fontweight='bold',
        bbox=dict(boxstyle='round', fc='#d5f5e3', alpha=0.9))
ax.set_xlabel('K Value  (left = complex model, right = simple model)', fontsize=12)
ax.set_ylabel('Error Rate', fontsize=12)
ax.set_title('The Bias-Variance Tradeoff in KNN\n'
             'Test error forms a U-shape — the bottom is our ideal K',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('09_bias_variance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best K = {best_k_bv}.')
print('Left zone:  Training error near 0 but test error is high -> memorised training data.')
print('Right zone: Both errors are high -> model is too simple.')
print('Middle:     Test error is lowest -> model generalises to new data.')


In [ ]:
# ── Cross-validation to find optimal K ───────────────────────────────────────
np.random.seed(42)
X_cv, y_cv = make_classification(n_samples=600, n_features=6,
                                  n_informative=4, n_redundant=2, random_state=42)
X_cv_sc    = StandardScaler().fit_transform(X_cv)
k_cv       = range(1, 41)
cv_mean    = []
cv_std     = []
for k in k_cv:
    s = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                        X_cv_sc, y_cv, cv=10)
    cv_mean.append(s.mean())
    cv_std.append(s.std())

cv_mean = np.array(cv_mean)
cv_std  = np.array(cv_std)
best_k_cv = list(k_cv)[np.argmax(cv_mean)]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(list(k_cv), cv_mean, 'b-o', ms=5, lw=2.5, label='Mean CV Accuracy (10 folds)')
ax.fill_between(list(k_cv), cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.2, color='blue', label='Plus/minus 1 Std Dev')
ax.axvline(x=best_k_cv, color='#e74c3c', ls='--', lw=2.5,
           label=f'Best K = {best_k_cv}  (CV Accuracy = {max(cv_mean):.4f})')
ax.scatter([best_k_cv], [cv_mean[best_k_cv-1]], color='red', s=200, zorder=5)
ax.set_xlabel('K Value', fontsize=12)
ax.set_ylabel('Cross-Validation Accuracy', fontsize=12)
ax.set_title(f'Finding Optimal K via 10-Fold Cross-Validation\n'
             f'Shaded band = stability of each K across different folds',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('10_cross_validation.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best K = {best_k_cv}  |  CV Accuracy = {max(cv_mean):.4f}')
print()
print('Why cross-validation?')
print('  One train/test split can get lucky or unlucky.')
print('  10-fold CV averages over 10 different splits -> much more reliable.')


---

# PART 9 — Pros and Cons

## 9.1 The Honest Assessment

KNN is one of the most elegant algorithms in machine learning — but elegance comes with real trade-offs. Let us go through each strength and weakness honestly, with live demonstrations.

---

### Pros

**1. Zero Training Time (Lazy Learner)**
When you call `knn.fit(X, y)`, nothing mathematical happens. The data is just stored. You can add new training examples instantly without retraining. Real-time, streaming data systems love this.

**2. No Distribution Assumptions (Non-Parametric)**
Logistic regression assumes a linear boundary. Naive Bayes assumes Gaussian distributions. KNN assumes nothing. This makes it universally applicable — weird-shaped data is handled naturally.

**3. Multi-Class is Free**
Other algorithms need modifications for multi-class (one-vs-all etc.). KNN handles 2, 5, or 100 classes with zero modification. It is just votes.

**4. Highly Interpretable**
A doctor using KNN can say: "Your results are most similar to patients X, Y, and Z, all of whom had condition A — so you likely have condition A too." That is auditable, explainable AI.

**5. Handles Non-Linear Boundaries**
As shown with the Circles dataset — KNN can model any shape. No linear algorithm can do that without complex feature engineering.

---

### Cons

**1. Slow Prediction (The Biggest Problem)**
Training is free, but you pay at prediction time. Every prediction requires comparing the new point to every single training example. With 1 million training points and 100 features, that is 100 million distance computations per prediction. This is O(n * d) complexity — it scales very badly.

**2. High Memory Requirements**
The entire training set must be stored forever. As data grows, so does RAM usage.

**3. Curse of Dimensionality**
In high-dimensional spaces (many features), something strange happens: all points become approximately equally distant from each other. The concept of "nearest" becomes meaningless. This is the most subtle and serious theoretical problem with KNN.

**4. Imbalanced Classes**
If 90% of training data is Class A, most neighbours will be Class A regardless of the true local structure. KNN has a built-in bias towards majority classes.

**5. Sensitive to Irrelevant Features**
Irrelevant features add noise to every distance calculation. If you are classifying jazz fans, features like shoe size or car colour will corrupt every distance and pick the wrong neighbours.


In [ ]:
# ── Pros and Cons visualised with live demos ─────────────────────────────────
fig = plt.figure(figsize=(19, 12))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('KNN Pros and Cons - Demonstrated with Data',
             fontsize=17, fontweight='bold')

# ── 1. Prediction time grows with training set size ───────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
sizes      = [200, 500, 1000, 2000, 4000, 7000]
pred_times = []
for n in sizes:
    Xt, yt = make_classification(n_samples=n+50, n_features=10, random_state=42)
    Xt = StandardScaler().fit_transform(Xt)
    m  = KNeighborsClassifier(n_neighbors=5)
    m.fit(Xt[:n], yt[:n])
    t0 = time.time()
    m.predict(Xt[n:])
    pred_times.append((time.time() - t0) * 1000)
ax1.plot(sizes, pred_times, 'r-o', ms=8, lw=2.5)
ax1.fill_between(sizes, pred_times, alpha=0.15, color='red')
for s, t in zip(sizes, pred_times):
    ax1.text(s, t+0.2, f'{t:.1f}ms', ha='center', fontsize=8)
ax1.set_xlabel('Training Set Size'); ax1.set_ylabel('Prediction Time (ms)')
ax1.set_title('CON: Slow Prediction\nGrows linearly with training data  O(n*d)', fontsize=10)
ax1.grid(True, alpha=0.3)

# ── 2. Curse of dimensionality ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
dims     = [2, 5, 10, 20, 50, 100, 200]
dim_accs = []
for d in dims:
    Xd, yd = make_classification(n_samples=400, n_features=d,
                                  n_informative=min(d, 5),
                                  n_redundant=0, random_state=42)
    s = cross_val_score(KNeighborsClassifier(5), StandardScaler().fit_transform(Xd),
                        yd, cv=5).mean()
    dim_accs.append(s)
ax2.plot(dims, dim_accs, 'purple', marker='D', ms=8, lw=2.5)
ax2.fill_between(dims, dim_accs, min(dim_accs)-0.02, alpha=0.15, color='purple')
for d, a in zip(dims, dim_accs):
    ax2.text(d, a+0.008, f'{a:.2f}', ha='center', fontsize=8)
ax2.set_xlabel('Number of Features (Dimensions)')
ax2.set_ylabel('KNN Accuracy')
ax2.set_title('CON: Curse of Dimensionality\nAccuracy collapses as features increase', fontsize=10)
ax2.grid(True, alpha=0.3)

# ── 3. Non-linear boundaries: KNN vs Logistic Regression ─────────────────────
ax3 = fig.add_subplot(gs[0, 2])
from sklearn.linear_model import LogisticRegression
np.random.seed(42)
Xc, yc = make_circles(n_samples=250, noise=0.1, factor=0.4, random_state=42)
Xc_sc  = StandardScaler().fit_transform(Xc)
knn_c  = KNeighborsClassifier(n_neighbors=7)
lr_c   = LogisticRegression()
knn_c.fit(Xc_sc, yc); lr_c.fit(Xc_sc, yc)
res = 0.025
x0m, x0x = Xc_sc[:,0].min()-0.5, Xc_sc[:,0].max()+0.5
x1m, x1x = Xc_sc[:,1].min()-0.5, Xc_sc[:,1].max()+0.5
xx3, yy3  = np.meshgrid(np.arange(x0m,x0x,res), np.arange(x1m,x1x,res))
Zk = knn_c.predict(np.c_[xx3.ravel(), yy3.ravel()]).reshape(xx3.shape)
Zl = lr_c.predict(np.c_[xx3.ravel(), yy3.ravel()]).reshape(xx3.shape)
ax3.contourf(xx3, yy3, Zk, alpha=0.3, cmap=ListedColormap(['#FFCCCC','#CCCCFF']))
ax3.contour(xx3, yy3, Zl, colors='darkgreen', linewidths=2.5, linestyles='--',
            levels=[0.5])
ax3.scatter(Xc_sc[yc==0,0], Xc_sc[yc==0,1], c='#cc0000', s=45, edgecolors='w', lw=0.5)
ax3.scatter(Xc_sc[yc==1,0], Xc_sc[yc==1,1], c='#0000cc', s=45, edgecolors='w', lw=0.5)
ax3.set_title(f'PRO: Non-Linear Boundaries\n'
              f'KNN (bg): {knn_c.score(Xc_sc,yc):.1%}  vs  '
              f'LogReg (green line): {lr_c.score(Xc_sc,yc):.1%}', fontsize=10)
ax3.set_xlabel('Feature 1'); ax3.set_ylabel('Feature 2')

# ── 4. Imbalanced classes ──────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
np.random.seed(7)
n_maj, n_min = 450, 50
X_imb = np.vstack([np.random.randn(n_maj, 2)*1.5,
                   np.random.randn(n_min, 2)*0.7 + [2, 2]])
y_imb = np.array([0]*n_maj + [1]*n_min)
X_imb_sc = StandardScaler().fit_transform(X_imb)
clf_imb  = KNeighborsClassifier(n_neighbors=9)
clf_imb.fit(X_imb_sc, y_imb)
plot_boundary(ax4, X_imb_sc, y_imb, clf_imb,
              'CON: Class Imbalance',
              f'Majority class n={n_maj} dominates minority n={n_min}')

# ── 5. Irrelevant features hurt accuracy ──────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
np.random.seed(0)
X_base, y_noise = make_moons(n_samples=300, noise=0.2, random_state=0)
noise_counts = [0, 2, 5, 10, 20, 40]
noise_accs   = []
for nc in noise_counts:
    Xn = np.hstack([X_base, np.random.randn(300, nc)]) if nc > 0 else X_base
    a  = cross_val_score(KNeighborsClassifier(5),
                         StandardScaler().fit_transform(Xn), y_noise, cv=5).mean()
    noise_accs.append(a)
ax5.plot(noise_counts, noise_accs, 'darkorange', marker='s', ms=8, lw=2.5)
ax5.fill_between(noise_counts, noise_accs, min(noise_accs)-0.01,
                 alpha=0.15, color='orange')
for nc, a in zip(noise_counts, noise_accs):
    ax5.text(nc, a+0.008, f'{a:.2f}', ha='center', fontsize=8)
ax5.set_xlabel('Irrelevant Noise Features Added')
ax5.set_ylabel('KNN Accuracy')
ax5.set_title('CON: Irrelevant Features Kill Accuracy\nEach noise feature corrupts distances', fontsize=10)
ax5.grid(True, alpha=0.3)

# ── 6. Lazy learner: instant adaptation ───────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
np.random.seed(42)
X_lazy    = np.random.randn(100, 2)
y_lazy    = (X_lazy[:,0] + X_lazy[:,1] > 0).astype(int)
X_lazy_sc = StandardScaler().fit_transform(X_lazy)
new_pts   = np.random.randn(20, 2)*0.5 + [2.5, 2.5]
new_pts_sc = StandardScaler().fit_transform(new_pts)
ax6.scatter(X_lazy_sc[y_lazy==0,0], X_lazy_sc[y_lazy==0,1],
            c='#cc0000', s=40, alpha=0.6, label='Class 0 (original)')
ax6.scatter(X_lazy_sc[y_lazy==1,0], X_lazy_sc[y_lazy==1,1],
            c='#0000cc', s=40, alpha=0.6, label='Class 1 (original)')
ax6.scatter(new_pts_sc[:,0], new_pts_sc[:,1],
            c='lime', s=120, marker='*', edgecolors='black', lw=1,
            zorder=5, label='New data added instantly!')
ax6.set_title('PRO: Instant Adaptation\nNew points added without retraining', fontsize=10)
ax6.set_xlabel('Feature 1'); ax6.set_ylabel('Feature 2')
ax6.legend(fontsize=8); ax6.grid(True, alpha=0.3)

plt.savefig('11_pros_cons.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top-left:  Prediction time grows with n - bad for large datasets.')
print('Top-mid:   Accuracy collapses in high dimensions.')
print('Top-right: KNN separates circles correctly; Logistic Regression fails.')
print('Bot-left:  Minority class barely gets any decision region.')
print('Bot-mid:   40 noise features drop accuracy significantly.')
print('Bot-right: New star points are instantly usable without retraining.')


---

# PART 10 — Best Practices

## 10.1 What Every KNN Practitioner Should Know

You now understand KNN deeply. Let us close with a practical framework for using it well.

---

### Pre-Flight Checklist

```
DATA PREPARATION
  [ ] Scale features with StandardScaler or MinMaxScaler
  [ ] Handle missing values (KNN cannot process NaN)
  [ ] Encode categorical features numerically
  [ ] Consider removing or imputing outliers
  [ ] Check for severe class imbalance -> consider oversampling

FEATURE SELECTION
  [ ] Remove obviously irrelevant features
  [ ] If more than 20 features, consider PCA or feature selection first
  [ ] Domain knowledge can tell you which features actually matter

CHOOSING K
  [ ] Start with K = sqrt(n_train) as a baseline
  [ ] Use odd K for binary classification to avoid ties
  [ ] Use 5-fold or 10-fold cross-validation to find the best K
  [ ] Try weights = 'distance' -- usually improves performance

DISTANCE METRIC
  [ ] Default: Euclidean (p=2) -- good for most continuous data
  [ ] Try Manhattan (p=1) if you have outliers
  [ ] Cosine for text or sparse high-dimensional data

PERFORMANCE
  [ ] If dataset > 50,000 rows, use algorithm = 'ball_tree' or 'kd_tree'
  [ ] These data structures reduce prediction time from O(n) to O(log n)
```

---

### When Should You Use KNN?

| Situation | Use KNN? | Reason |
|-----------|----------|--------|
| Small to medium dataset under 50k rows | Yes | Fast enough |
| Non-linear or unknown boundary shape | Yes | KNN adapts to any shape |
| Need to explain individual predictions | Yes | "Similar to patients X and Y" |
| New training data arrives constantly | Yes | No retraining needed |
| Very large dataset over 500k rows | No | Too slow for prediction |
| Many features (>50) | No | Curse of dimensionality |
| Real-time predictions at scale | No | O(n*d) per prediction is too slow |
| Many irrelevant features | No | Corrupts distance calculations |


In [ ]:
# ── Production-ready KNN template ─────────────────────────────────────────────
# Copy and adapt this for any classification task

np.random.seed(42)
X_demo, y_demo = make_moons(n_samples=500, noise=0.25, random_state=42)

# Step 1: Split BEFORE scaling -- prevents data leakage
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=42
)

# Step 2: Scale -- fit on training only
scaler_d  = StandardScaler()
X_tr_d_sc = scaler_d.fit_transform(X_tr_d)   # fit + transform
X_te_d_sc = scaler_d.transform(X_te_d)        # transform only

# Step 3: Find best K via cross-validation
print('Searching for best K ...')
best_k_d, best_cv_d = 1, 0.0
for k in range(1, 31):
    cv = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, weights='distance'),
        X_tr_d_sc, y_tr_d, cv=5
    ).mean()
    if cv > best_cv_d:
        best_k_d, best_cv_d = k, cv
print(f'  Best K = {best_k_d}  (CV score = {best_cv_d:.4f})')
print()

# Step 4: Train final model
final_knn = KNeighborsClassifier(
    n_neighbors  = best_k_d,
    weights      = 'distance',   # closer neighbours matter more
    metric       = 'euclidean',  # or 'manhattan'
    algorithm    = 'auto'        # sklearn picks the fastest structure
)
final_knn.fit(X_tr_d_sc, y_tr_d)

# Step 5: Evaluate
y_pred_d = final_knn.predict(X_te_d_sc)
print('=' * 52)
print(f'  Final Model  KNN  K={best_k_d}  weights=distance')
print('=' * 52)
print(f'  Test Accuracy : {accuracy_score(y_te_d, y_pred_d)*100:.2f}%')
print()
print(classification_report(y_te_d, y_pred_d))

# Step 6: Predict a new unseen point
new_raw = np.array([[0.5, -0.3]])
new_sc  = scaler_d.transform(new_raw)        # MUST scale using the same scaler
pred    = final_knn.predict(new_sc)
proba   = final_knn.predict_proba(new_sc)
print(f'  New point -> Class {pred[0]}')
print(f'  Probabilities: Class 0 = {proba[0][0]:.2%}  |  Class 1 = {proba[0][1]:.2%}')


In [ ]:
# ── Final dark-theme summary card ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor('#0d1117')
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor('#0d1117'); ax.axis('off')

ax.text(0.5, 0.96, 'K-Nearest Neighbors - Complete Lecture Summary',
        ha='center', va='top', fontsize=24, fontweight='bold',
        color='white', transform=ax.transAxes)

CARDS = [
    (0.02, 0.88, 0.30, 0.32, '#58a6ff', '#161b22', 'The Algorithm',
     ['1. Store ALL training data (no fitting)',
      '2. New point arrives for prediction',
      '3. Compute distance to every training point',
      '4. Sort -> take the K nearest',
      '5a. Classification -> majority VOTE',
      '5b. Regression    -> AVERAGE values',
      '6. Return prediction']),
    (0.35, 0.88, 0.30, 0.32, '#3fb950', '#161b22', 'Distance Formulas',
     ['Euclidean (p=2): sqrt(sum(A-B)^2)  default',
      'Manhattan (p=1): sum|Ai-Bi|  robust',
      'Minkowski (p=?): tune p as hyperparameter',
      'Chebyshev (p->inf): max|Ai-Bi|',
      'Cosine: 1-(A.B)/(||A||*||B||)  text data',
      '',
      'ALWAYS scale features before KNN!']),
    (0.68, 0.88, 0.30, 0.32, '#d2a8ff', '#161b22', 'Choosing K',
     ['Small K -> Low bias, High variance (overfit)',
      'Large K -> High bias, Low variance (underfit)',
      '',
      'Start: K = sqrt(n_training_samples)',
      'Use odd K for binary (avoids ties)',
      'Find K via cross-validation',
      'Use weights="distance" by default']),
    (0.02, 0.50, 0.30, 0.32, '#ffa657', '#161b22', 'Pros',
     ['Zero training time (lazy learner)',
      'No distribution assumptions needed',
      'Multi-class works out of the box',
      'Highly interpretable predictions',
      'Non-linear boundaries for free',
      'Instantly adapts to new data',
      'Excellent baseline model']),
    (0.35, 0.50, 0.30, 0.32, '#f85149', '#161b22', 'Cons',
     ['Slow prediction: O(n * d) per query',
      'Stores full training set in memory',
      'Curse of dimensionality (d > 20)',
      'Biased towards majority class',
      'Irrelevant features corrupt distances',
      'No feature importance scores',
      'No model compression possible']),
    (0.68, 0.50, 0.30, 0.32, '#79c0ff', '#161b22', 'Use KNN When',
     ['Dataset is small to medium (<50k)',
      'Non-linear boundary expected',
      'Predictions need to be explainable',
      'New training data arrives often',
      'Need a quick, solid baseline',
      '',
      'Avoid: large data, many features, real-time']),
]

for (x, y, w, h, tc, bg, title, lines) in CARDS:
    patch = mpatches.FancyBboxPatch(
        (x, y-h), w, h, boxstyle='round,pad=0.008',
        facecolor=bg, edgecolor=tc, linewidth=2,
        transform=ax.transAxes)
    ax.add_patch(patch)
    ax.text(x+w/2, y-0.018, title,
            ha='center', va='top', fontsize=12.5, fontweight='bold',
            color=tc, transform=ax.transAxes)
    ax.axhline(y=y-0.055, xmin=x+0.005, xmax=x+w-0.005,
               color=tc, lw=0.8, alpha=0.5, transform=ax.transAxes)
    for i, line in enumerate(lines):
        ax.text(x+0.012, y-0.070-i*0.033, line,
                ha='left', va='top', fontsize=9,
                color='#c9d1d9', transform=ax.transAxes,
                fontfamily='monospace')

ax.text(0.5, 0.06,
        '"Tell me who your nearest neighbours are, and I will tell you who you are."  --  The KNN Philosophy',
        ha='center', va='bottom', fontsize=12, color='#8b949e',
        style='italic', transform=ax.transAxes)

plt.savefig('12_summary_card.png', dpi=140, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Lecture complete!')
print('Topics covered:')
for t in ['What KNN is and how it thinks',
          'Distance formulas and why scaling matters',
          'Classification with majority voting',
          'Regression with averaging',
          'Weighted KNN for better decisions',
          'Decision boundaries and non-linearity',
          'Bias-Variance tradeoff through K',
          'Pros and Cons with live demonstrations',
          'Best practices and production template']:
    print(f'  {t}')
